# 1. Initialization

In [0]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(repo_root)

print(f"[INFO] Repo root added to path: {repo_root}")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

from functools import reduce

from common.helpers import get_table, get_bronze

print("[INFO] License Allocations Silver pipeline started")

try:
    dbutils.widgets.text("batch_id", "")
    batch_id = dbutils.widgets.get("batch_id")
except:
    batch_id = None

print("[INFO] Databricks Spark session ready")
spark
print(f"[INFO] Batch ID: {batch_id}")

# License Allocations Silver Pipeline

Grain: one row per `allocation_id` + `allocation_date`

## 2. Read Bronze license_allocations

In [0]:
bronze_path = "/Volumes/datalake_catalog/datalake_schema/bronze/license_allocations"

df_bronze = get_bronze(bronze_path, spark=spark)
df_bronze.show(5)
df_bronze.printSchema()

In [0]:
df1 = df_bronze.drop("dw_ingested_at", "dw_source_file", "dw_batch_id", "batch_id", "source_table", "ingest_time")

## 3. Cast Types

In [0]:
df2 = (
    df1.select(
        F.col("allocation_id").cast("bigint"),
        F.col("license_id").cast("bigint"),
        F.col("seat_number").cast("int"),
        F.col("status").cast("string"),
        F.col("allocation_date").cast("timestamp"),
        F.col("ingest_time").cast("timestamp")
    )
)

## 4. Validate Status

In [0]:
valid_status = ["active", "deactivated"]

df3 = df2.filter(F.col("status").isin(valid_status))
df3_quarantine = df2.filter(~F.col("status").isin(valid_status))

## 5. Deduplicate

In [0]:
w = Window.partitionBy("allocation_id", "allocation_date").orderBy(F.col("ingest_time").desc())

df_ranked = df3.withColumn("rn", F.row_number().over(w))

df4 = df_ranked.filter(F.col("rn") == 1).drop("rn")

df4_quarantine = (
    df_ranked.filter(F.col("rn") > 1)
    .drop("rn")
    .withColumn("validation_error", F.lit("duplicate allocation_id"))
)

## 6. Validate license_id + seat range

In [0]:
silver_license_path = "/Volumes/datalake_catalog/datalake_schema/silver/licenses"

df_license_ref = (
    spark.read.format("delta").load(silver_license_path)
).select("license_id")

df5 = (
    df4.join(df_license_ref, on="license_id", how="left_semi")
    .filter(F.col("seat_number").between(1, 550))
)

df5_quarantine = (
    df4.join(df_license_ref, on="license_id", how="left_anti")
    .unionByName(
        df4.filter(~F.col("seat_number").between(1, 550)),
        allowMissingColumns=True
    )
)

## 7. Combine quarantine

In [0]:
df_quarantine_all = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    [df3_quarantine, df4_quarantine, df5_quarantine]
)

## 8. Prepare Silver Table

In [0]:
df_silver = df5

## 9. Upsert to Silver Delta

In [0]:
silver_path = "/Volumes/datalake_catalog/datalake_schema/silver/license_allocations"

cols = df_silver.columns

df_upsert = df_silver

w = Window.partitionBy("allocation_id", "allocation_date").orderBy(F.col("ingest_time").desc())
df_upsert = df_upsert.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn")

if DeltaTable.isDeltaTable(spark, silver_path):
    print("[INFO] Updating existing silver table")
    target = DeltaTable.forPath(spark, silver_path)

    (
        target.alias("t")
        .merge(df_upsert.alias("s"), "t.allocation_id = s.allocation_id")
        .whenMatchedUpdate(set={c: f"s.{c}" for c in cols if c != "allocation_id"})
        .whenNotMatchedInsert(values={c: f"s.{c}" for c in cols})
        .execute()
    )
else:
    print("[INFO] Creating silver table")
    df_upsert.write.format("delta").mode("overwrite").save(silver_path)

# 10. Verification

In [0]:
spark.read.format("delta").load(silver_path).show(5)